# NPPE-2: Multilingual Speech Recognition
**Roll Number:** 21F3000028

**Name:** Anuj Dev Singh

**Task:** Transcribe multilingual speech (English, Hindi, Tamil) from audio

**Model:** OpenAI Whisper Large-V3

**Metric:** Word Error Rate (WER) — lower is better

**Approach:**
- Whisper Large-V3 is a 1.5B parameter multilingual ASR model pretrained on 680K hours of audio
- Natively supports English, Hindi, and Tamil with strong zero-shot performance
- Beam search decoding (num_beams=5) for higher transcription accuracy
- Auto language detection handles multilingual audio without manual labeling
- Transcribes in original scripts (Devanagari, Tamil) rather than translating to English

**Design choice — inference-only:**
- Whisper Large-V3 already achieves strong WER on all 3 target languages
- Fine-tuning 1.5B params exceeds Kaggle T4 GPU memory
- Fine-tuning smaller variants degraded multilingual performance
- With 2000 samples, overfitting risk outweighs gains

## 1. Environment Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/multilingual-speech-recognition/sample_submission.csv
/kaggle/input/competitions/multilingual-speech-recognition/train.csv
/kaggle/input/competitions/multilingual-speech-recognition/test.csv
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00000.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00019.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00088.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00071.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00084.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00001.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00031.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00094.wav
/kaggl

In [3]:
import subprocess, sys

for pkg in ["jiwer", "librosa", "soundfile", "evaluate", "accelerate"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", pkg])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


## 2. Imports

In [4]:
import gc, re, random, warnings, torch, librosa
from pathlib import Path
from dataclasses import dataclass
from jiwer import wer
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from datasets import Dataset
import evaluate

warnings.filterwarnings("ignore")

## 3. HuggingFace Authentication

In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except:
    pass

## 4. Configuration

In [6]:
class Config:
    INPUT_DIR    = Path("/kaggle/input")
    OUTPUT_DIR   = Path("/kaggle/working")
    MODEL_SAVE   = OUTPUT_DIR / "whisper-finetuned"
    SR           = 16000
    MODEL_NAME   = "openai/whisper-large-v3"
    EPOCHS       = 3
    BATCH_SIZE   = 4
    GRAD_ACCUM   = 4
    LEARNING_RATE = 1e-5
    WARMUP_STEPS = 50
    MAX_LABEL_LEN = 448
    SEED         = 42

random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: Tesla T4


## 5. Locate Data Files

In [7]:
input_root = Config.INPUT_DIR
comp_dirs = list(input_root.iterdir())
print(f"Input dirs: {comp_dirs}")

train_csv = None
test_csv = None
audio_dir = None

for root, dirs, files in os.walk(str(input_root)):
    for f in files:
        fp = os.path.join(root, f)
        if f == 'train.csv':
            train_csv = fp
        elif f == 'test.csv':
            test_csv = fp
        elif f.endswith('.wav') and audio_dir is None:
            audio_dir = root

print(f"train.csv: {train_csv}")
print(f"test.csv:  {test_csv}")
print(f"Audio dir: {audio_dir}")

if audio_dir is None:
    for root, dirs, files in os.walk(str(input_root)):
        if 'competition_data' in dirs:
            audio_dir = os.path.join(root, 'competition_data')
        for d in dirs:
            full = os.path.join(root, d)
            wavs = [x for x in os.listdir(full) if x.endswith('.wav')]
            if len(wavs) > 10:
                audio_dir = full
                break
    print(f"Audio dir (2nd pass): {audio_dir}")

Input dirs: [PosixPath('/kaggle/input/competitions')]
train.csv: /kaggle/input/competitions/multilingual-speech-recognition/train.csv
test.csv:  /kaggle/input/competitions/multilingual-speech-recognition/test.csv
Audio dir: /kaggle/input/competitions/multilingual-speech-recognition/competition_data/test


## 6. Load and Explore Data

In [8]:
train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Train columns: {train_df.columns.tolist()}")
print(f"Test columns:  {test_df.columns.tolist()}")
train_df.head()

Train: 2000 | Test: 100
Train columns: ['id', 'audio', 'text']
Test columns:  ['id', 'audio']


,id,audio,text
0,0,audio_00000.wav,you had quoted plutarch line.
1,1,audio_00001.wav,மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...
2,2,audio_00002.wav,to do his phd in engineering about four years ...
3,3,audio_00003.wav,maybe he was not at home.
4,4,audio_00004.wav,BUT WE DIDN'T BREAK HIS OLD WINDOW YOU KNOW EX...


## 7. Analyze Language Distribution

In [9]:
import re

def detect_lang(text):
    text = str(text)
    if re.search(r'[\u0B80-\u0BFF]', text):
        return 'ta'
    if re.search(r'[\u0900-\u097F]', text):
        return 'hi'
    return 'en'

train_df['lang'] = train_df['text'].apply(detect_lang)
print(train_df['lang'].value_counts())

upper_count = train_df['text'].apply(lambda x: str(x) == str(x).upper()).sum()
print(f"\nAll-uppercase entries: {upper_count}/{len(train_df)}")

lang
en    998
hi    508
ta    494
Name: count, dtype: int64

All-uppercase entries: 1502/2000


## 8. Verify Audio Files

In [10]:
sample_audio = train_df.iloc[0]['audio']
sample_path = os.path.join(audio_dir, sample_audio) if audio_dir else None

if sample_path and os.path.exists(sample_path):
    audio, sr = librosa.load(sample_path, sr=Config.SR)
    print(f"Loaded: {sample_path}")
    print(f"Duration: {len(audio)/sr:.2f}s | Sample rate: {sr}")
else:
    print(f"File not found: {sample_path}")
    print("Listing audio_dir contents...")
    if audio_dir:
        print(os.listdir(audio_dir)[:10])

Loaded: /kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00000.wav
Duration: 7.23s | Sample rate: 16000


## 9. Load Whisper Large-V3 Model

In [11]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(Config.MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(Config.MODEL_NAME, task="transcribe")
processor = WhisperProcessor.from_pretrained(Config.MODEL_NAME, task="transcribe")

model = WhisperForConditionalGeneration.from_pretrained(Config.MODEL_NAME)
model.config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.config.use_cache = False

print(f"Loaded: {Config.MODEL_NAME}")

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Loaded: openai/whisper-large-v3


## 10. Resolve Train/Test Audio Directories

In [12]:
train_audio_dir = None
test_audio_dir = None

for root, dirs, files in os.walk(str(Config.INPUT_DIR)):
    wavs = [f for f in files if f.endswith('.wav')]
    if len(wavs) > 0:
        if 'train' in root:
            train_audio_dir = root
        elif 'test' in root:
            test_audio_dir = root
        else:
            if train_audio_dir is None:
                train_audio_dir = root

if train_audio_dir is None:
    train_audio_dir = audio_dir
if test_audio_dir is None:
    test_audio_dir = audio_dir

print(f"Train audio: {train_audio_dir}")
print(f"Test audio:  {test_audio_dir}")

sample_train = os.path.join(train_audio_dir, train_df.iloc[0]['audio'])
sample_test = os.path.join(test_audio_dir, test_df.iloc[0]['audio'])
print(f"Train file exists: {os.path.exists(sample_train)}")
print(f"Test file exists:  {os.path.exists(sample_test)}")

Train audio: /kaggle/input/competitions/multilingual-speech-recognition/competition_data/train
Test audio:  /kaggle/input/competitions/multilingual-speech-recognition/competition_data/test
Train file exists: True
Test file exists:  True


## 11. Transcription Function
Beam search decoding with auto language detection.

In [13]:
model = model.to(device)

def transcribe(audio_file, is_test=False):
    folder = test_audio_dir if is_test else train_audio_dir
    path = os.path.join(folder, audio_file)
    audio, sr = librosa.load(path, sr=Config.SR)
    inputs = processor(audio, sampling_rate=Config.SR, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch.float16)
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            max_new_tokens=440,
            num_beams=5,
            do_sample=False,
            task="transcribe",
            condition_on_prev_tokens=False,
            compression_ratio_threshold=None,
            no_speech_threshold=None,
            return_timestamps=True,
        )
    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text.strip()

## 12. Validation
Check WER on a sample of training data.

In [14]:
def detect_language_from_audio(audio_file, is_test=False):
    folder = test_audio_dir if is_test else train_audio_dir
    path = os.path.join(folder, audio_file)
    audio, sr = librosa.load(path, sr=Config.SR)

    inputs = processor(
        audio,
        sampling_rate=Config.SR,
        return_tensors="pt",
    )
    input_features = inputs.input_features.to(device, dtype=torch.float16)

    with torch.no_grad():
        lang_ids = model.detect_language(input_features)

    if isinstance(lang_ids, tuple):
        lang_ids = lang_ids[0]

    if isinstance(lang_ids, list) and len(lang_ids) > 0:
        if isinstance(lang_ids[0], dict):
            detected = max(lang_ids[0], key=lang_ids[0].get)
            detected = detected.replace('<|', '').replace('|>', '')
            return detected
    return 'en'


test_lang = detect_language_from_audio(train_df.iloc[0]['audio'])
print(f"Test detection: {test_lang} (actual: {train_df.iloc[0].get('lang', 'unknown')})")


Test detection: en (actual: en)


In [15]:
val_sample = train_df.sample(n=20, random_state=42).reset_index(drop=True)

val_preds = []
val_refs = []

for idx, row in val_sample.iterrows():
    pred = transcribe(row['audio'], is_test=False)
    ref = str(row['text']).strip()
    val_preds.append(pred.lower())
    val_refs.append(ref.lower())
    print(f"{idx+1}/20: {pred[:60]}")

from jiwer import wer
val_wer = wer(val_refs, val_preds)
print(f"\nValidation WER: {val_wer:.4f}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'trans

1/20: இவங்களுடன் வாழ்க்கை முறைகளும் வித்தியாசமாக இருக்கும் அந்த மா


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2/20: Killed Gogoi during the night.


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

18/20: वह कभी खाने पर नियंतर नहीं कर पाता है


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


19/20: இதுக்கு அவசரமே கொஸி இந்த மாதிரி வந்துவிட்டால் டிங்கு இந்த மா


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


20/20: यहीं से इंदिरा गंधी की राजनितिक वापसी की शुरुआत हुई।

Validation WER: 0.2316


## 13. Generate Test Predictions

In [16]:
test_preds = []

for idx, row in test_df.iterrows():
    try:
        pred = transcribe(row['audio'], is_test=True)
    except Exception as e:
        print(f"Error on {row['audio']}: {e}")
        pred = ""
    test_preds.append(pred)
    if (idx + 1) % 10 == 0:
        print(f"  {idx + 1}/{len(test_df)}")

print(f"Done. {len(test_preds)} predictions.")

Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10/100


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  20/100


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  80/100


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  90/100


Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=440) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  100/100
Done. 100 predictions.


## 14. Create Submission

In [17]:
submission = pd.DataFrame({
    "audio": test_df["audio"],
    "text": test_preds,
})

assert list(submission.columns) == ["audio", "text"]
assert len(submission) == len(test_df)

submission.to_csv(Config.OUTPUT_DIR / "submission.csv", index=False)
print(f"Saved ({len(submission)} rows)")
print(submission.head(10).to_string(index=False))

Saved (100 rows)
          audio                                                                                                                                                                                                                                                                       text
audio_00000.wav                                                                                                                                                                                                                 इन में जैपी, मोर्गन, सिम्मस और ब्लैक रोक के सीएयो आमिन हैं
audio_00001.wav                                                                                                                                                                                                                   ایک ہزار نو سو نبی میں جہاں دو مسلم امیدوانوں نے جیتا ہے
audio_00002.wav                                                                                                மாட்டுப்பங்கள் முடிந்து